# Práctica 3 · Etapa 4 — Modelos de regresión y ajuste de hiperparámetros

**3201 Modelos Inteligentes de Aprendizaje / 529 Redes Neuronales Modulares** · Maestría y Doctorado en Ciencias e Ingeniería (MyDCI) · Facultad de Ingeniería, Arquitectura y Diseño ·
Universidad Autónoma de Baja California · Ciclo 2026-2 · Versión 6

**Profesores:** Dr. Everardo Inzunza González · Dr. Enrique Efrén García Guerrero · Dr. Oscar Adrián Aguirre Castro

**Fase 4 de la sesión** · presupuesto: **30 min**

---

Tres regresores de familias distintas aprenden la misma tabla: **XGBoost** (árboles construidos en secuencia, cada uno corrigiendo los residuos del anterior), **SVR** (regresión de vectores de soporte con núcleo RBF) y **Random Forest** (árboles independientes que se promedian). Para cada uno se explora una rejilla de hiperparámetros midiendo el error en **validación**; la mejor combinación se reentrena con entrenamiento + validación y produce, una sola vez, el pronóstico del bloque de prueba. Las métricas de prueba se calculan en la Etapa 5.

| Necesita | Produce |
|---|---|
| `artefactos/caracteristicas_2038.csv` y `artefactos/particion_2038.json` (Etapas 2 y 3) | Tabla G · `artefactos/hiperparametros_2038.json`, `predicciones_2038.csv` y `rejilla_2038.csv` |

**Cómo leer este cuaderno.** Cada bloque abre con una celda de texto que explica *qué* se hace y *por qué*. Dentro del
código, el comentario a la derecha de cada instrucción dice para qué sirve esa línea. Las salidas marcadas con
**`← Anote`** son las que usted copia en la tabla indicada del manual; el resto sólo se ejecuta y se observa.
Ejecute las celdas **en orden**; si cambia algo, vuelva a ejecutar desde esa celda hacia abajo.

---
## Bloque 0 · Puesta en marcha

Esta celda se ejecuta **siempre primero**, en cualquier entorno. Detecta si está en Google Colab, en el JupyterHub del
laboratorio o en un Jupyter local; localiza la carpeta de la práctica (`P3_SeriesTiempo_v6`, la que descargó de
Classroom); crea `resultados/` con sus subcarpetas sin borrar nada; instala sólo lo que falte; fija la semilla e imprime
las versiones con que se ejecutó. Reproducir un resultado empieza por saber con qué se obtuvo.

> **En Colab**, cada cuaderno corre en su propia máquina virtual: los archivos que un cuaderno guarda sólo los ve el
> siguiente si viven en **Google Drive**. Por eso la carpeta de la práctica se sube a *Mi unidad* y esta celda la monta.

In [ ]:
# ── Bloque 0 · Puesta en marcha (Etapa 4) ─────────────────────────────────────────────
import sys, os, json, time, hashlib, warnings, subprocess, importlib  # utilidades estándar de Python
from pathlib import Path                                   # rutas que funcionan igual en Windows, Linux y Colab
T0 = time.perf_counter()                                   # cronómetro de este cuaderno
warnings.filterwarnings("ignore", category=FutureWarning)  # oculta sólo avisos de versiones futuras

try:
    import google.colab                                    # este módulo sólo existe dentro de Google Colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

CARPETA_PRACTICA = "P3_SeriesTiempo_v6"                    # la carpeta que descargó de Classroom
if EN_COLAB:
    from google.colab import drive                         # acceso a Google Drive desde Colab
    drive.mount("/content/drive")                          # la primera vez pide autorización
    RAIZ_DRIVE = Path("/content/drive/MyDrive")            # «Mi unidad»
    candidatas = [RAIZ_DRIVE / CARPETA_PRACTICA, RAIZ_DRIVE / "Colab Notebooks" / CARPETA_PRACTICA]
    candidatas += sorted(RAIZ_DRIVE.glob(f"*/{CARPETA_PRACTICA}"))  # también un nivel más abajo
else:
    aqui = Path.cwd().resolve()                            # carpeta desde donde se abrió el cuaderno
    candidatas = [aqui, *aqui.parents][:4]                 # el cuaderno vive en notebooks/: se busca hacia arriba
    candidatas += [p / CARPETA_PRACTICA for p in candidatas]
BASE = next((c for c in candidatas if (c / "datos").is_dir() and (c / "notebooks").is_dir()), None)
if BASE is None:
    raise FileNotFoundError("No encuentro la carpeta de la práctica (con datos/ y notebooks/). "
                            "Revise la Sección 4.3 del manual: en Colab súbala completa a «Mi unidad».")
DATOS, RES = BASE / "datos", BASE / "resultados"           # entradas y salidas de la práctica
FIG, TAB, ART = RES / "figuras", RES / "tablas", RES / "artefactos"  # figuras · tablas · archivos intermedios
for carpeta in (FIG, TAB, ART):
    carpeta.mkdir(parents=True, exist_ok=True)             # se crean si faltan; nunca se borra nada

# Bibliotecas: se instala sólo lo que falte (en Colab casi todo viene preinstalado).
NECESARIAS = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "xgboost": "xgboost",
               "matplotlib": "matplotlib", "seaborn": "seaborn", "statsmodels": "statsmodels",
               "openpyxl": "openpyxl", "scipy": "scipy"}
for modulo, paquete in NECESARIAS.items():
    try:
        importlib.import_module(modulo)                    # ¿ya está instalado?
    except ImportError:
        print("Instalando", paquete, "…")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

import numpy as np                                         # cálculo numérico con arreglos
import pandas as pd                                        # tablas de datos indexadas por fecha
import matplotlib                                          # motor de gráficas
import matplotlib.pyplot as plt                            # interfaz de gráficas
import sklearn, xgboost as xgb                             # scikit-learn y XGBoost
import seaborn, statsmodels, scipy                         # sólo para registrar sus versiones aquí

def _ver(v):
    return tuple(int("".join(ch for ch in p if ch.isdigit()) or 0) for p in v.split(".")[:2])
assert _ver(sklearn.__version__) >= (1, 4), "Se necesita scikit-learn ≥ 1.4 (root_mean_squared_error)."
assert _ver(xgb.__version__) >= (2, 0), "Se necesita XGBoost ≥ 2.0 (early_stopping_rounds en el constructor)."

SEMILLA = 42                                               # una sola semilla para toda la práctica
np.random.seed(SEMILLA)                                    # fija el azar de numpy
pd.set_option("display.width", 160)                        # tablas anchas sin cortes de renglón
pd.set_option("display.max_columns", 30)
COLORES = {"real": "#15241E", "XGBoost": "#17795C", "SVR": "#D9A21B", "Random Forest": "#9E2B25",
            "Ingenuo": "#9AA7A1", "Ingenuo estacional": "#5E6E67", "Climatología": "#165542"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})  # estilo uniforme de las figuras

def guardar(fig, nombre):
    """Guarda la figura en resultados/figuras/ (PNG, 150 dpi) para insertarla en el manual."""
    ruta = FIG / f"{nombre}.png"
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    print("Figura guardada →", ruta.relative_to(BASE))

def _serializable(v):
    """Convierte tipos de numpy y pandas a tipos de Python; los no finitos (NaN, ±inf) quedan como None."""
    if isinstance(v, dict):
        return {str(k): _serializable(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [_serializable(x) for x in v]
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, (float, np.floating)):
        return float(v) if np.isfinite(v) else None
    if isinstance(v, pd.Timestamp):
        return v.isoformat()
    return v

def registrar(clave, valor):
    """Añade un resultado a resultados/registro_P3.json (se acumula entre cuadernos; JSON estricto, sin NaN)."""
    ruta = RES / "registro_P3.json"
    reg = json.loads(ruta.read_text(encoding="utf-8")) if ruta.exists() else {}
    reg[clave] = _serializable(valor)
    ruta.write_text(json.dumps(reg, ensure_ascii=False, indent=1, allow_nan=False), encoding="utf-8")

def md5(ruta):
    return hashlib.md5(Path(ruta).read_bytes()).hexdigest()  # huella del archivo

if EN_COLAB:
    ENTORNO = "Google Colab"
elif "JUPYTERHUB_USER" in os.environ:
    ENTORNO = "JupyterHub"                                 # sesión abierta en el JupyterHub del laboratorio
elif "ai_env" in sys.executable:
    ENTORNO = "Kernel ai_env del laboratorio"              # el mismo kernel, fuera de una sesión del Hub
else:
    ENTORNO = "Jupyter local"
VERSIONES = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
              "scikit-learn": sklearn.__version__, "xgboost": xgb.__version__, "matplotlib": matplotlib.__version__,
              "seaborn": seaborn.__version__, "statsmodels": statsmodels.__version__, "scipy": scipy.__version__}
registrar("etapa4_entorno", {"entorno": ENTORNO, **VERSIONES})
print("Entorno        :", ENTORNO)                         # ← Anote (Tabla A)
print("Carpeta base   :", BASE)
for k, v in VERSIONES.items():
    print(f"{k:<15}:", v)                                  # ← Anote (Tabla A): Python, scikit-learn, XGBoost y pandas

**Reconstrucción de la partición.** Se vuelve a aplicar exactamente la misma partición de la Etapa 3 y se comprueba,
**fecha por fecha**, que la lista completa de meses de cada bloque coincide con la guardada en `particion_2038.json`.
Si no coincide, algo cambió en las etapas anteriores.

In [ ]:
from sklearn.model_selection import train_test_split
for nombre in ("caracteristicas_2038.csv", "particion_2038.json"):
    if not (ART / nombre).exists():
        raise FileNotFoundError(f"Falta {nombre}: ejecute primero las Etapas 2 y 3.")
tabla = pd.read_csv(ART / "caracteristicas_2038.csv", index_col="FECHA", parse_dates=True)
y = tabla.pop("y"); X = tabla                     # objetivo y características
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, shuffle=False)  # igual que en la Etapa 3
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, shuffle=False)
guardada = json.loads((ART / "particion_2038.json").read_text(encoding="utf-8"))
for nombre, parte in {"entrenamiento": y_train, "validación": y_val, "prueba": y_test}.items():
    fechas = [f"{t:%Y-%m}" for t in parte.index]  # todos los meses del bloque, no sólo el primero
    assert fechas == guardada[nombre]["fechas"], f"El bloque «{nombre}» no coincide con el de la Etapa 3"
X_trval, y_trval = pd.concat([X_train, X_val]), pd.concat([y_train, y_val])  # para el reentrenamiento final
print("Partición idéntica a la de la Etapa 3 (fecha por fecha):", len(y_train), len(y_val), len(y_test))

---
## 4.1 Los tres modelos y sus rejillas

| Modelo | Escalado | Hiperparámetros explorados | Qué controla cada uno |
|---|---|---|---|
| XGBoost | no lo necesita (árboles) | `n_estimators`, `max_depth`, `learning_rate` | número de árboles · profundidad de cada uno · tamaño del paso con que cada árbol corrige al anterior |
| SVR (RBF) | **sí**: X con `StandardScaler` en un `Pipeline`, y además `y` con `TransformedTargetRegressor` | `C`, `gamma`, `epsilon` | castigo a los errores (C grande = menos regularización) · alcance de cada punto del núcleo · ancho del tubo dentro del cual el error no cuenta |
| Random Forest | no lo necesita | `n_estimators`, `max_depth`, `min_samples_leaf` | número de árboles · profundidad · tamaño mínimo de cada hoja (suaviza) |

El escalado va **dentro** del `Pipeline` para que sus medias y desviaciones se aprendan sólo con los datos de ajuste
de cada paso, nunca con validación o prueba. XGBoost no forma parte de scikit-learn: es una biblioteca independiente
con interfaz compatible (`fit` / `predict`), por eso se combina sin fricción con las herramientas de scikit-learn.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

MODELOS = {
    "XGBoost": (XGBRegressor(objective="reg:squarederror", tree_method="hist",  # error cuadrático; histogramas
                             random_state=SEMILLA, n_jobs=-1),
                {"n_estimators": [100, 300], "max_depth": [2, 3, 5], "learning_rate": [0.03, 0.1]}),
    "SVR": (TransformedTargetRegressor(  # escala también la y (mm) para que epsilon tenga sentido
                regressor=make_pipeline(StandardScaler(), SVR(kernel="rbf")), transformer=StandardScaler()),
            {"regressor__svr__C": [0.1, 1, 10], "regressor__svr__gamma": ["scale", 0.01, 0.1],
             "regressor__svr__epsilon": [0.05, 0.1, 0.3]}),
    "Random Forest": (RandomForestRegressor(random_state=SEMILLA, n_jobs=-1),  # n_jobs=-1 usa todos los núcleos
                      {"n_estimators": [200, 500], "max_depth": [3, 5, None], "min_samples_leaf": [1, 5]}),
}
for nombre, (_, rejilla) in MODELOS.items():
    print(f"{nombre:<14}: {len(ParameterGrid(rejilla)):3d} combinaciones")  # ← Anote (Tabla G)

---
## 4.2 Búsqueda en la rejilla con el conjunto de validación

Para cada combinación: se ajusta con **entrenamiento**, se predice **validación** y se mide el RMSE. Gana la de menor
RMSE de validación. El conjunto de prueba **no participa**: si se usara para elegir, su error dejaría de ser una
estimación honesta del desempeño en datos futuros.

In [ ]:
resultados_rejilla, mejores, pred_val = [], {}, {}
for nombre, (estimador, rejilla) in MODELOS.items():
    t0 = time.perf_counter()
    for params in ParameterGrid(rejilla):
        m = clone(estimador).set_params(**params).fit(X_train, y_train)  # ajusta con entrenamiento
        p = m.predict(X_val)      # predice validación
        resultados_rejilla.append({"modelo": nombre, **{k.split("__")[-1]: v for k, v in params.items()},
                                   "RMSE_val": root_mean_squared_error(y_val, p),
                                   "MAE_val": mean_absolute_error(y_val, p)})
        if nombre not in mejores or resultados_rejilla[-1]["RMSE_val"] < mejores[nombre]["RMSE_val"]:
            mejores[nombre] = {"params": params, "RMSE_val": resultados_rejilla[-1]["RMSE_val"],
                               "MAE_val": resultados_rejilla[-1]["MAE_val"]}
            pred_val[nombre] = p  # guarda la predicción de la mejor combinación
    mejores[nombre]["t_rejilla_s"] = time.perf_counter() - t0  # tiempo de toda la búsqueda
rejilla_df = pd.DataFrame(resultados_rejilla)
rejilla_df.sort_values(["modelo", "RMSE_val"]).groupby("modelo").head(3).round(3)  # las 3 mejores de cada modelo

In [ ]:
for nombre, r in mejores.items():
    print(f"{nombre:<14} mejores hiperparámetros: {r['params']}")  # ← Anote (Tabla G)
    print(f"{'':<14} RMSE de validación = {r['RMSE_val']:.2f} mm · MAE = {r['MAE_val']:.2f} mm · "
          f"búsqueda en {r['t_rejilla_s']:.1f} s")  # ← Anote (Tabla G)
elegido_val = min(mejores, key=lambda n: mejores[n]["RMSE_val"])  # regla de decisión: menor RMSE de validación
print(f"\nModelo elegido en validación (menor RMSE): {elegido_val} · n de validación = {len(y_val)}")  # ← Anote (Tabla G)

> **Decisión metodológica 2 · Dos decisiones distintas: cómo se eligen los hiperparámetros y con qué datos se ajusta
> el modelo final.** Son independientes y conviene no mezclarlas.
>
> *Procedimiento de selección.* Aquí se usa **un solo** bloque de validación (hold-out): cada combinación se ajusta con
> entrenamiento y se mide en validación. La alternativa es la validación cruzada temporal (`GridSearchCV` con
> `TimeSeriesSplit`), que mide cada combinación en varios orígenes y promedia; se usa en la Etapa 6.
>
> *Ventana de ajuste del modelo final.* Una vez elegidos los hiperparámetros, la validación ya cumplió su función de
> selección; reentrenar con entrenamiento + validación da al modelo los años inmediatamente anteriores a la prueba,
> los más parecidos al futuro que se va a pronosticar. El costo es que el modelo final no es exactamente el que se
> validó (más datos, mismos hiperparámetros). La otra opción —ajustar sólo con entrenamiento— evalúa el modelo
> validado tal cual, a cambio de desperdiciar los datos más recientes.
>
> **Justifique en su reporte** qué combinación de las dos decisiones usaría para su propio problema.

---
## 4.3 Reentrenamiento final y pronóstico del bloque de prueba (una sola vez)

In [ ]:
finales, pred_test, pred_trval, t_ajuste = {}, {}, {}, {}
for nombre, (estimador, _) in MODELOS.items():
    t0 = time.perf_counter()
    finales[nombre] = clone(estimador).set_params(**mejores[nombre]["params"]).fit(X_trval, y_trval)  # reentrena
    t_ajuste[nombre] = time.perf_counter() - t0            # tiempo de un ajuste final
    pred_test[nombre] = finales[nombre].predict(X_test)    # el pronóstico que se evaluará en la Etapa 5
    pred_trval[nombre] = finales[nombre].predict(X_trval)  # ajuste dentro de muestra (para diagnosticar sobreajuste)
    print(f"{nombre:<14} ajuste final en {t_ajuste[nombre]:.2f} s")  # ← Anote (Tabla G)

In [ ]:
import json
hp = {n: {"params": {k: (v if not isinstance(v, (np.integer, np.floating)) else v.item())
                     for k, v in mejores[n]["params"].items()},
          "combinaciones": len(ParameterGrid(MODELOS[n][1])),
          "RMSE_val": round(mejores[n]["RMSE_val"], 4), "MAE_val": round(mejores[n]["MAE_val"], 4),
          "t_rejilla_s": round(mejores[n]["t_rejilla_s"], 3), "t_ajuste_s": round(t_ajuste[n], 3)} for n in MODELOS}
hp["elegido_en_validacion"] = elegido_val                 # el modelo que la regla de decisión lleva a la prueba
hp["n_validacion"] = int(len(y_val))
(ART / "hiperparametros_2038.json").write_text(json.dumps(hp, ensure_ascii=False, indent=1), encoding="utf-8")
pd.DataFrame({"real": y_test, **pred_test}).to_csv(ART / "predicciones_2038.csv", index_label="FECHA")
pd.DataFrame({"real": y_trval, **pred_trval}).to_csv(ART / "predicciones_trval_2038.csv", index_label="FECHA")
pd.DataFrame({"real": y_val, **pred_val}).to_csv(ART / "predicciones_val_2038.csv", index_label="FECHA")
rejilla_df.to_csv(ART / "rejilla_2038.csv", index=False)
rejilla_df.to_csv(TAB / "rejilla_2038.csv", index=False)  # copia para el reporte
registrar("etapa4", hp)
print("Guardados: hiperparametros_2038.json, predicciones_2038.csv, predicciones_trval_2038.csv, "
      "predicciones_val_2038.csv y rejilla_2038.csv")

---
## Cierre de la Etapa 4

> **Punto de verificación.** La Tabla G tiene, para cada modelo, el número de combinaciones, los hiperparámetros elegidos, el RMSE y el MAE de validación y los dos tiempos (búsqueda y ajuste final). Compare el RMSE de validación de cada modelo con el de la climatología en validación (Tabla F): ¿alguno la supera con claridad? Anote cuál modelo tiene el menor RMSE de validación: ése es el que se «lleva» a la prueba según la regla de decisión de la Etapa 5, fijada antes de abrir el bloque reservado.

**Siguiente:** abra `P3_E5_Metricas_Curvas_Comparacion.ipynb`.

In [ ]:
minutos = (time.perf_counter() - T0) / 60  # tiempo de pared desde el Bloque 0
registrar("etapa4_duracion_min", round(minutos, 2))
print(f"Etapa 4 terminada en {minutos:.1f} min · resultados en", RES.relative_to(BASE))  # ← Anote (Tabla A): duración